# 최적 균일도 Recipe 좌표 연산

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn import set_config
set_config(display='text')

## CCD 기반 Dry Etch 실험 데이터 정의
- x1 : Temperature
- x2 : Gas Flow
- 결과 : Selectivity

In [2]:
data = pd.DataFrame({
    "x1": [0, -1, 1, -1, 1, -1.414, 1.414, 0, 0, 0, 0, 0],
    "x2": [0, -1, -1, 1, 1, 0, 0, -1.414, 1.414, 0, 0, 0],
    "Selectivity": [12.5, 8.2, 9.8, 7.5, 11.2, 8.9, 11.5, 7.8, 12.1, 12.6, 12.4, 12.7]
})

## 2차 다항식에 들어갈 feature 생성
- x1, x2 기반 [x1, x2, x1x2, $x1^2$, $x2^2$

In [3]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(data[["x1", "x2"]])

## Regression Model Fitting
- 실제 2차항 기반 모델 피팅

In [4]:
model = LinearRegression()
model.fit(X_poly, data["Selectivity"])

LinearRegression()

## 행렬 미분 통한 극점 기반 최적점 도출
$$y=β_0​+β_1​x_1​+β_2​x_2​+β_{11}​x_{12}​+β_{22}​x_{22}​+β_{12}​x_1​x_2​$$

$$
\nabla y=
\begin{bmatrix}
\frac{\partial y}{\partial x_1}\\
\frac{\partial y}{\partial x_2}
\end{bmatrix}
=
\begin{bmatrix}
\beta_1\\
\beta_2
\end{bmatrix}
+
\begin{bmatrix}
2\beta_{11} & \beta_{12}\\
\beta_{12} & 2\beta_{22}
\end{bmatrix}
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
=\mathbf{0}
$$

$$
\nabla y=\mathbf{b}+2\mathbf{B}\mathbf{x}
$$

$$
\mathbf{b}=
\begin{bmatrix}
\beta_1\\
\beta_2
\end{bmatrix},
\qquad
\mathbf{x}=
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix},
\qquad
\mathbf{B}=
\begin{bmatrix}
\beta_{11} & \beta_{12}/2\\
\beta_{12}/2 & \beta_{22}
\end{bmatrix}
$$

$$
\mathbf{x}^*
=
-\frac{1}{2}\mathbf{B}^{-1}\mathbf{b}
$$

In [10]:
b_vec = np.array([model.coef_[0], model.coef_[1]])
B_mat = np.array([
    [model.coef_[2], model.coef_[3]/2],
    [model.coef_[3]/2, model.coef_[4]]
])
x_opt = -0.5 * np.linalg.inv(B_mat).dot(b_vec)

print("--- Estimated Beta Coefficients ---")
print(f"b1(Temp) = {b_vec[0]:.4f}, b2(Gas) = {b_vec[1]:.4f}")
print(f"Optimal Temp Code (x1_opt) = {x_opt[0]:.4f}")
print(f"Optimal Gas Code (x2_opt) = {x_opt[1]:.4f}")

--- Estimated Beta Coefficients ---
b1(Temp) = 1.1222, b2(Gas) = 0.8477
Optimal Temp Code (x1_opt) = 0.4679
Optimal Gas Code (x2_opt) = 0.3584
